# Create zarr stroring unstitched tiles

Ome-zarr libs:
https://github.com/ome/ome-zarr-py/issues/407

In [ ]:
import glob
import numpy as np
from ome_zarr.image import NgffImage, NgffMultiscales
from ome_zarr.scene import NgffScene
from ome_zarr_models._v06.coordinate_transforms import CoordinateSystem, CoordinateSystemIdentifier, Axis, Translation
import os.path
import tifffile

## Source data

In [ ]:
source_path = 'C:/Project/slides/example_cells/*.tiff'

filenames = glob.glob(source_path)
filenames

## Store as zarr

In [ ]:
pyramid_downsample = 2
npyramid_add = 2
zarr_version = 3
ome_version = '0.6'
pixel_size = {'y': 0.004, 'x': 0.004}
pixel_size_list = np.array(list(pixel_size.values()))
output_path = '../output/tiles.ome.zarr'
dim_order='cyx'


def read_tiff(filename):
    tif = tifffile.TiffFile(filename)
    data = tif.asarray()
    data = rgba_to_rgb(data)
    data = np.moveaxis(data, -1, 0)
    metadata = tags_to_dict(tif.pages.first.tags)
    translation = [convert_rational_value(metadata.get('XPosition')),
                   convert_rational_value(metadata.get('YPosition'))]
    return data, translation


def tags_to_dict(tags):
    tag_dict = {}
    for tag in tags.values():
        tag_dict[tag.name] = tag.value
    return tag_dict


def convert_rational_value(value):
    if value is not None and isinstance(value, tuple):
        if value[0] == value[1]:
            value = value[0]
        else:
            value = value[0] / value[1]
    return value


def rgba_to_rgb(data):
    alpha = np.atleast_3d(data[:, :, 3]) / np.float32(255)
    return (data[:, :, :3] * alpha).astype(np.uint8)


def init_cs():
    cs_world = CoordinateSystem(
        name="stitched",
        axes=(
            Axis(name="c", type="channel"),
            Axis(name="y", type="space", unit="micrometer"),
            Axis(name="x", type="space", unit="micrometer"),
        )
    )
    return cs_world


def write_tile(path, data, dim_order, pixel_size, name):
    scale_factors = [pyramid_downsample ** (scale + 1) for scale in range(npyramid_add)]

    ngff_image = NgffImage(data, dim_order, pixel_size, {'x': 'micrometer', 'y': 'micrometer'}, name=name)
    ngff_multiscales = NgffMultiscales(ngff_image, scale_factors)

    ngff_multiscales.to_ome_zarr(path)

    return ngff_multiscales


def create_coordinate_transform(multiscales, translation):
    transform = Translation(
        translation=translation,
        input=CoordinateSystemIdentifier(
                path=multiscales.name,
                name=multiscales.metadata.coordinateSystems[0].name
            ),
        output=CoordinateSystemIdentifier(name="stitched")
    )
    return transform


def write_scene(path, images, transforms, cs_world):
    ngff_scene = NgffScene(
        images=images,
        transformations=transforms,
        coordinate_systems=(cs_world,)
    )

    ngff_scene.to_ome_zarr(path)

## Store as zarr

In [ ]:
cs = init_cs()

images = []
transforms = []
for filename in filenames:
    data, metadata = read_tiff(filename)
    tile_name = os.path.splitext(os.path.basename(filename))[0]
    path = f'{output_path}/{tile_name}'
    print(f'{filename} -> {path}', data.shape)
    translation = np.array(list(map(int, tile_name.split('_')[:2]))) * data.shape[1:] * pixel_size_list
    translation = [0] + list(translation)
    multiscales = write_tile(path, data, dim_order, pixel_size, tile_name)
    images.append(multiscales)
    transforms.append(create_coordinate_transform(multiscales, translation))
print(metadata)
write_scene(output_path, images, transforms, cs)
